In [35]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("public/data")          # adjust to your layout
INFILE = DATA_DIR / "datav2.xlsx"   # <-- change to your actual filename
OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [36]:
df = pd.read_excel(
    INFILE,
    sheet_name="in",
    engine="openpyxl"
)

print("Shape:", df.shape)
df.head()

Shape: (154, 13)


,Organization Name,Nest,Org ID,organization type,Geographic Area,verified,Notes,Primary,2ndry,AE's finds -->,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,Air National Guard,NaN,NaN,Emergency Management,Federal,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,American Society of Civil Engineers,NaN,ASCE,Integrative Research,Regional,Y,NaN,?,NaN,NaN,NaN,NaN,NaN
2,BC Hydro,NaN,BC Hydro,Infrastructure and Planning,British Columbia,N,NaN,Joseph.Farrugia@bchydro.com,martin.lawrence11@yahoo.com,NaN,NaN,NaN,NaN
3,BNSF,NaN,BNSF,Infrastructure and Planning,Regional,Y,rail system in Canada,NaN,https://www.bnsf.com/about-bnsf/contact-us-form.page,NaN,NaN,NaN,NaN
4,Bonneville Power Administration,NaN,BPA,Infrastructure and Planning,Federal,N,NaN,jgnguyen@bpa.gov,NaN,NaN,NaN,NaN,NaN


## look into Org ID

In [37]:
# Identify rows missing Org ID
dropped_rows = df[df["Org ID"].isna()]

print("Number of rows that will be dropped:", len(dropped_rows))

# List the organization names
print("\nOrganization Names being dropped:")
display(
    dropped_rows["Organization Name"]
    .value_counts(dropna=False)
)

# Now actually drop them
df = df.dropna(subset=["Org ID"])

print("\nShape after dropping missing Org ID:", df.shape)

Number of rows that will be dropped: 21

Organization Names being dropped:


NaN                                             10
Air National Guard                               1
California State Parks                           1
Canada's Insurance Agency                        1
Canada's Insurance Regulators                    1
City of Vancouver Department of Public Works     1
Federal Media                                    1
School Districts                                 1
SCL?                                             1
Sitka Sound Science Center                       1
Washington National Guard                        1
Washington State Parks                           1
Name: Organization Name, dtype: int64


Shape after dropping missing Org ID: (133, 13)


In [38]:
dup_orgid = df[df.duplicated("Org ID", keep=False)]

print("Number of rows with duplicated Org ID:", len(dup_orgid))
display(dup_orgid.sort_values("Org ID"))

Number of rows with duplicated Org ID: 0


,Organization Name,Nest,Org ID,organization type,Geographic Area,verified,Notes,Primary,2ndry,AE's finds -->,Unnamed: 10,Unnamed: 11,Unnamed: 12


In [39]:
# remove all whitespace inside Org ID
df["Org ID"] = (
    df["Org ID"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

display(df[["Organization Name", "Org ID"]].head(20))

,Organization Name,Org ID
1,American Society of Civil Engineers,ASCE
2,BC Hydro,BCHydro
3,BNSF,BNSF
4,Bonneville Power Administration,BPA
5,British Columbia Emergency Management,BCEM
6,British Columbia Government - BC Emergency Alert System,BCEAS
7,British Columbia Government - GeoBC,GeoBC
8,Bureau of Land Management,BLM
9,CA Seismic Safety Commission,CaSSC
10,Cal Fire,CalFire


## Organization type

In [40]:
print("Unique Organization Types:", df["organization type"].nunique())
print()

display(
    df["organization type"]
    .value_counts(dropna=False)
)

Unique Organization Types: 17



Infrastructure and Planning                                   32
Emergency Management                                          30
Earthquake and Hazard Science                                 25
Integrative Research                                          19
Community Organization                                         5
Infrastructure and Planning                                    5
Media                                                          4
Response                                                       2
NaN                                                            2
Emergency Management, Community Organization                   1
Earthquake and Hazard Science /Infrastucture                   1
Academic, Integrative Research                                 1
???                                                            1
Earthquake and Hazard Science, Emergency Management            1
FreeChoiceLearning, Integrative Research                       1
emergency Management     

In [41]:
df["org_type_clean"] = (
    df["organization type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)
df["org_type_clean"].unique()

array(['integrative research', 'infrastructure and planning',
       'emergency management', 'earthquake and hazard science',
       'earthquake and hazard science /infrastucture', 'media',
       'emergency management, community organization',
       'community organization', 'nan', 'response',
       'academic, integrative research', '???',
       'earthquake and hazard science, emergency management',
       'freechoicelearning, integrative research', 'insurance',
       'earthquake and hazard science, infrastructure and planning'],
      dtype=object)

In [42]:
CANON = {
    "integrative research": "Integrative Research",
    "infrastructure and planning": "Infrastructure and Planning",
    "emergency management": "Emergency Management",
    "earthquake and hazard science": "Earthquake and Hazard Science",
    "media": "Media",
    "community organization": "Community Organization",
    "response": "Response",
    "academic": "Academic",
    "freechoicelearning": "FreeChoiceLearning",
    "insurance": "Insurance",
    "unknown": "Unknown",
}

# map known weird tokens to canonical keys (lowercase keys)
TOKEN_FIX = {
    "earthquake and hazard science /infrastucture": "earthquake and hazard science, infrastructure and planning",
    # in case you see other misspellings later:
    "infrastucture": "infrastructure and planning",
    "infrastructure": "infrastructure and planning",
    "???": "unknown",
    "nan": "unknown",
    "": "unknown",
}

In [43]:
def normalize_org_types(s: str):
    if s is None:
        s = ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)

    # fix whole-string known issues first
    s = TOKEN_FIX.get(s, s)

    # treat "/" as a separator like ","
    s = s.replace("/", ",")

    # split on commas
    toks = [t.strip() for t in s.split(",") if t.strip()]

    # fix per-token issues (typos like "infrastucture")
    fixed = []
    for t in toks:
        t = TOKEN_FIX.get(t, t)
        fixed.append(t)

    # map to canonical labels (drop unknown tokens into "Unknown")
    mapped = []
    for t in fixed:
        mapped.append(CANON.get(t, "Unknown"))

    # de-dup while preserving order
    out = []
    for m in mapped:
        if m not in out:
            out.append(m)

    if not out:
        out = ["Unknown"]

    primary = out[0]
    return out, primary

df[["orgTypes", "orgTypePrimary"]] = df["org_type_clean"].apply(
    lambda s: pd.Series(normalize_org_types(s))
)

df[["Organization Name", "organization type", "org_type_clean", "orgTypePrimary", "orgTypes"]].head(20)

,Organization Name,organization type,org_type_clean,orgTypePrimary,orgTypes
1,American Society of Civil Engineers,Integrative Research,integrative research,Integrative Research,[Integrative Research]
2,BC Hydro,Infrastructure and Planning,infrastructure and planning,Infrastructure and Planning,[Infrastructure and Planning]
3,BNSF,Infrastructure and Planning,infrastructure and planning,Infrastructure and Planning,[Infrastructure and Planning]
4,Bonneville Power Administration,Infrastructure and Planning,infrastructure and planning,Infrastructure and Planning,[Infrastructure and Planning]
5,British Columbia Emergency Management,Emergency Management,emergency management,Emergency Management,[Emergency Management]
6,British Columbia Government - BC Emergency Alert System,Emergency Management,emergency management,Emergency Management,[Emergency Management]
7,British Columbia Government - GeoBC,Earthquake and Hazard Science,earthquake and hazard science,Earthquake and Hazard Science,[Earthquake and Hazard Science]
8,Bureau of Land Management,Earthquake and Hazard Science /Infrastucture,earthquake and hazard science /infrastucture,Earthquake and Hazard Science,"[Earthquake and Hazard Science, Infrastructure and Planning]"
9,CA Seismic Safety Commission,Integrative Research,integrative research,Integrative Research,[Integrative Research]
10,Cal Fire,Emergency Management,emergency management,Emergency Management,[Emergency Management]


In [44]:
# how many primaries
display(df["orgTypePrimary"].value_counts())

# what are all possible labels across lists
all_labels = sorted({lab for labs in df["orgTypes"] for lab in labs})
print("All canonical org type labels:")
for x in all_labels:
    print("-", x)

# rows that still look suspicious (contain Unknown)
display(df[df["orgTypes"].apply(lambda xs: "Unknown" in xs)][
    ["Org ID","Organization Name","organization type","org_type_clean","orgTypes"]
].head(50))

Infrastructure and Planning      37
Emergency Management             32
Earthquake and Hazard Science    28
Integrative Research             19
Community Organization            5
Media                             4
Unknown                           3
Response                          2
Academic                          1
FreeChoiceLearning                1
Insurance                         1
Name: orgTypePrimary, dtype: int64

All canonical org type labels:
- Academic
- Community Organization
- Earthquake and Hazard Science
- Emergency Management
- FreeChoiceLearning
- Infrastructure and Planning
- Insurance
- Integrative Research
- Media
- Response
- Unknown


,Org ID,Organization Name,organization type,org_type_clean,orgTypes
35,OCMP,DLCD - Oregon Coastal Management Program,NaN,nan,[Unknown]
48,FIN,Finance Canada,NaN,nan,[Unknown]
54,ImageCAT,ImageCAT,???,???,[Unknown]


## geo cleaning

In [45]:
df["geo_clean"] = (
    df["Geographic Area"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# Treat string "nan" as missing (happens after astype(str))
df.loc[df["geo_clean"].isin(["nan", "none", "n/a", ""]), "geo_clean"] = ""

In [46]:
print("Unique Geographic Areas (raw):", df["Geographic Area"].nunique(dropna=True))
print("Unique Geographic Areas (clean):", df.loc[df["geo_clean"] != "", "geo_clean"].nunique())
print()

print("Counts (raw):")
display(df["Geographic Area"].value_counts(dropna=False))

print("Counts (clean):")
display(df["geo_clean"].replace("", pd.NA).value_counts(dropna=False))

Unique Geographic Areas (raw): 12
Unique Geographic Areas (clean): 10

Counts (raw):


Oregon              35
Federal             32
California          16
Regional            15
Washington          14
British Columbia    11
International        4
Canada               2
Regional (OR WA)     1
Federal              1
Alaska               1
Washington           1
Name: Geographic Area, dtype: int64

Counts (clean):


oregon              35
federal             33
california          16
regional            15
washington          15
british columbia    11
international        4
canada               2
regional (or wa)     1
alaska               1
Name: geo_clean, dtype: int64

In [47]:
ALLOWED_GEOS = {
    "oregon": "Oregon",
    "federal": "Federal",
    "california": "California",
    "regional": "Regional",
    "washington": "Washington",
    "british columbia": "British Columbia",
    "international": "International",
    "canada": "Canada",
    "alaska": "Alaska",
}

# variants -> canonical key
GEO_FIX = {
    "": "unknown",
    "nan": "unknown",
    "n/a": "unknown",
    "none": "unknown",
    "???": "unknown",
    "regional (or wa)": "regional",
    "regional (or wa )": "regional",
    "regional (or, wa)": "regional",
}

def geo_canonical(s):
    s = "" if s is None else str(s).strip().lower()
    s = " ".join(s.split())
    s = GEO_FIX.get(s, s)

    if s in ALLOWED_GEOS:
        return ALLOWED_GEOS[s]
    if s == "unknown":
        return "Unknown"
    # anything unexpected: keep visible so you notice it
    return "Unknown"

df["geoPrimary"] = df["geo_clean"].apply(geo_canonical)
df["geoTags"] = df["geoPrimary"].apply(lambda g: [g])

display(df["geoPrimary"].value_counts())

Oregon              35
Federal             33
Regional            16
California          16
Washington          15
British Columbia    11
International        4
Canada               2
Alaska               1
Name: geoPrimary, dtype: int64

In [48]:
df.head(3)

,Organization Name,Nest,Org ID,organization type,Geographic Area,verified,Notes,Primary,2ndry,AE's finds -->,Unnamed: 10,Unnamed: 11,Unnamed: 12,org_type_clean,orgTypes,orgTypePrimary,geo_clean,geoPrimary,geoTags
1,American Society of Civil Engineers,NaN,ASCE,Integrative Research,Regional,Y,NaN,?,NaN,NaN,NaN,NaN,NaN,integrative research,[Integrative Research],Integrative Research,regional,Regional,[Regional]
2,BC Hydro,NaN,BCHydro,Infrastructure and Planning,British Columbia,N,NaN,Joseph.Farrugia@bchydro.com,martin.lawrence11@yahoo.com,NaN,NaN,NaN,NaN,infrastructure and planning,[Infrastructure and Planning],Infrastructure and Planning,british columbia,British Columbia,[British Columbia]
3,BNSF,NaN,BNSF,Infrastructure and Planning,Regional,Y,rail system in Canada,NaN,https://www.bnsf.com/about-bnsf/contact-us-form.page,NaN,NaN,NaN,NaN,infrastructure and planning,[Infrastructure and Planning],Infrastructure and Planning,regional,Regional,[Regional]


In [49]:
import json

# ensure Org ID behaves as a string identifier
df["Org ID"] = df["Org ID"].astype(str)

# encode list column for safe CSV export
df["orgTypes_json"] = df["orgTypes"].apply(json.dumps)

cols = [
    "Organization Name",
    "Org ID",
    "orgTypes_json",
    "orgTypePrimary",
    "geoPrimary",
    "Notes",
    "Primary",
    "2ndry",
]

clean_df = df[cols].copy()

clean_df.to_csv("organizations_clean.csv", index=False)

print("Exported:", len(clean_df), "rows")
clean_df.head()

Exported: 133 rows


,Organization Name,Org ID,orgTypes_json,orgTypePrimary,geoPrimary,Notes,Primary,2ndry
1,American Society of Civil Engineers,ASCE,"[""Integrative Research""]",Integrative Research,Regional,NaN,?,NaN
2,BC Hydro,BCHydro,"[""Infrastructure and Planning""]",Infrastructure and Planning,British Columbia,NaN,Joseph.Farrugia@bchydro.com,martin.lawrence11@yahoo.com
3,BNSF,BNSF,"[""Infrastructure and Planning""]",Infrastructure and Planning,Regional,rail system in Canada,NaN,https://www.bnsf.com/about-bnsf/contact-us-form.page
4,Bonneville Power Administration,BPA,"[""Infrastructure and Planning""]",Infrastructure and Planning,Federal,NaN,jgnguyen@bpa.gov,NaN
5,British Columbia Emergency Management,BCEM,"[""Emergency Management""]",Emergency Management,British Columbia,NaN,PreparedBC@gov.bc.ca,Aaron.Hinks@gov.bc.ca


# looking at edges

In [51]:
df_edges = pd.read_excel(
    INFILE,
    sheet_name="Relationships",
    engine="openpyxl"
)

print("Shape:", df.shape)
df_edges.head()

Shape: (536, 5)


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,NaN,NaN
1,BCHydro,NRCanGSC,tools/products,sharing models,NaN
2,NRCanGSC,BCHydro,data,NaN,NaN
3,NRCanGSC,BCHydro,tools/products,sharing models,NaN
4,CRESCENT,BCHydro,data,NaN,NaN


In [52]:
df_edges = df_edges.copy()

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    # remove all whitespace, then (optional) strip punctuation the same way as nodes
    s = str(x).strip()
    s = pd.Series([s]).str.replace(r"\s+", "", regex=True).iloc[0]
    return s

df_edges["From agency"] = df_edges["From agency"].apply(clean_id)
df_edges["To agency"]   = df_edges["To agency"].apply(clean_id)

# Optional: also normalize Relationship type / Description / Status whitespace
for c in ["Relationship type", "Description", "Status"]:
    if c in df_edges.columns:
        df_edges[c] = (df_edges[c].astype("string")
                                   .str.strip()
                                   .str.replace(r"\s+", " ", regex=True))

# Drop *exact* duplicate rows (keeps first occurrence)
before = len(df_edges)
df_edges_clean = df_edges.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df_edges_clean)

print(f"Rows before: {before}  |  after drop_duplicates: {after}  |  removed: {before-after}")
df_edges_clean.head()

Rows before: 536  |  after drop_duplicates: 520  |  removed: 16


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,<NA>,<NA>
1,BCHydro,NRCanGSC,tools/products,sharing models,<NA>
2,NRCanGSC,BCHydro,data,<NA>,<NA>
3,NRCanGSC,BCHydro,tools/products,sharing models,<NA>
4,CRESCENT,BCHydro,data,<NA>,<NA>


In [53]:
df_edges_clean.to_csv("edges_clean.csv", index=False)
print("Wrote edges_clean.csv")

Wrote edges_clean.csv


# verify alignment

In [54]:
node_ids = set(clean_df["Org ID"].astype(str))

missing_from = sorted(set(df_edges_clean["From agency"]) - node_ids)
missing_to   = sorted(set(df_edges_clean["To agency"]) - node_ids)

print("Missing From agency IDs (not in nodes):", missing_from[:50])
print("Missing To agency IDs (not in nodes):", missing_to[:50])

Missing From agency IDs (not in nodes): ['BCAlert', 'CAmedia', 'Copes', 'DNR', 'EERi', 'LidarBC', 'NHRP', 'NOAA-NTWC', 'NOAA-NWS', 'NOAANWS', 'NOAANWTC', 'ODHS', 'ORmedia', 'PBEM', 'PDXOEM', 'PWB', 'PacificCor', 'RDPO', 'Tribal', 'USGA', 'WAmedia', 'WPUC', 'copes', 'dogami', 'eeri', 'oem']
Missing To agency IDs (not in nodes): ['CAmedia', 'Copes', 'DNR', 'NHRP', 'NOAA-NTWC', 'NOAA-NWS', 'ODHS', 'ORmedia', 'Osspac', 'PBEM', 'PDXOEM', 'PWB', 'RDPO', 'Sz4d', 'Tribal', 'WAmedia', 'copes', 'dogami', 'oem']
